# Comparación de modelos

**Para qué sirve:** explorar a fondo los resultados de `scripts.train_models` y `scripts.evaluate_models`: ranking,
variabilidad entre folds, confusiones por clase, errores por segmento e importancia de features.

**Antes de usarlo:** ejecuta `python -m scripts.prepare_data`, `python -m scripts.train_models` y
`python -m scripts.evaluate_models`. Este cuaderno solo **lee** esos resultados.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

# Las rutas de config/ y data/ son relativas a la raíz del repositorio: nos movemos a ella.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Raíz del repo:", ROOT)

In [ ]:
MODELS_CM = ["svm", "mlp", "knn"]   # modelos para las matrices de confusión
MODEL_DETAIL = "svm"                # modelo para el análisis de errores
RUN_DIAGNOSIS = False               # True = recalcula honesta vs. K-Fold con fuga (tarda más)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown

from src.config import load_yaml
from src.dataset import load_dataset, feature_columns
from src.evaluation.compare_models import load_runs, comparison_table, per_class_f1
from src.models import load_model
from src.models.common import models_root

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
REPORTS = Path(load_yaml("experiment/settings.yaml")["paths"]["analysis"]) / "reports"

runs = load_runs()
table = comparison_table(runs)
classes = next(iter(runs.values()))["meta"]["classes"]
meta = next(iter(runs.values()))["meta"]
print(f"{meta['n_windows']} ventanas | {meta['n_features']} features | folds: {meta['folds']}")
display(table.round(3))

## F1 por clase
Las columnas más claras (valores bajos) son las clases que el modelo no logra separar.

In [ ]:
display(per_class_f1(runs).round(2).style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

## Variabilidad entre folds
Cada punto es el macro-F1 de un fold (con LOSO, un sujeto). Si los puntos de dos modelos se traslapan, **no se puede
afirmar que uno sea mejor que el otro**. La línea punteada es el azar.

In [ ]:
pf = pd.DataFrame({m: {f: v["macro_f1"] for f, v in r["per_fold"].items()} for m, r in runs.items()}).T
order = table["model"].tolist()
fig, ax = plt.subplots(figsize=(9, 0.62 * len(order) + 1.6))
for i, m in enumerate(order):
    ax.hlines(i, pf.loc[m].min(), pf.loc[m].max(), color="gray", lw=1)
    ax.scatter(pf.loc[m], [i] * pf.shape[1], s=70, zorder=3)
    for f, v in pf.loc[m].items():
        ax.annotate(f.split("=")[-1], (v, i), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7)
ax.axvline(1 / len(classes), ls="--", color="gray")
ax.set_yticks(range(len(order)), order); ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xlabel("macro-F1 por fold")
fig.tight_layout()

## Matrices de confusión
Normalizadas por fila (recall). La diagonal es lo que se acierta; fuera de ella, con qué clase se confunde.

In [ ]:
mods = [m for m in MODELS_CM if m in runs]
fig, axes = plt.subplots(1, len(mods), figsize=(5.2 * len(mods), 4.7), squeeze=False)
for ax, m in zip(axes[0], mods):
    cm = np.array(runs[m]["pooled"]["confusion_matrix"], float)
    norm = cm / cm.sum(axis=1, keepdims=True).clip(1)
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
    ax.set_xticks(range(len(classes)), classes, rotation=35, ha="right"); ax.set_yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f"{norm[i, j]:.0%}", ha="center", va="center",
                    color="white" if norm[i, j] > 0.55 else "black", fontsize=9)
    ax.set_title(f"{m} | macro-F1 {runs[m]['pooled']['macro_f1']:.2f}"); ax.set_xlabel("predicho"); ax.set_ylabel("real")
fig.tight_layout()

## Errores por sujeto y por segmento
- **Recall por fold y clase:** ¿hay un sujeto para el que el modelo falla mucho más?
- **Peores segmentos:** los gestos completos peor clasificados (la columna `acierto_ventanas` es la fracción de sus
  ventanas que se acertó). Sirve para ver si los errores son de todo el segmento o de unas pocas ventanas.

In [ ]:
pred = pd.read_csv(models_root() / MODEL_DETAIL / "predictions.csv")
pred["ok"] = pred["y_true_name"] == pred["y_pred_name"]

recall = pred.groupby(["fold", "y_true_name"])["ok"].mean().unstack().reindex(columns=classes).round(2)
display(recall.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).set_caption(f"Recall por fold y clase ({MODEL_DETAIL})"))

seg = pred.groupby(["subject", "segment_id"]).agg(
    trial=("trial", "first"), rep=("rep", "first"), real=("y_true_name", "first"),
    predicho_mayoria=("y_pred_name", lambda v: v.mode().iloc[0]),
    acierto_ventanas=("ok", "mean"), n_ventanas=("ok", "size")).reset_index()
display(seg.sort_values("acierto_ventanas").head(15).round(2))

## Importancia de features (Random Forest)
Qué features y qué canales usa el bosque para decidir. Es **descriptivo** (importancia por impureza), no causal, y
se calcula con el modelo final entrenado con todas las ventanas.

In [ ]:
rf_path = models_root() / "rf" / "model.joblib"
if not rf_path.exists():
    print("No hay modelo 'rf' guardado: ejecuta python -m scripts.train_models --models rf")
else:
    df, _ = load_dataset(with_raw=False)
    model, _meta = load_model(rf_path)
    imp = pd.Series(model.named_steps["clf"].feature_importances_, index=feature_columns(df)).sort_values(ascending=False)

    tipo = imp.index.str.replace(r"^EMG\d_", "", regex=True)
    canal = imp.index.str.extract(r"^(EMG\d)_")[0].fillna("cruzadas (grupos)")
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={"width_ratios": [1.5, 1, 0.8]})
    imp.head(15)[::-1].plot.barh(ax=axes[0], title="Top 15 features")
    imp.groupby(tipo.to_numpy()).sum().sort_values().plot.barh(ax=axes[1], title="Por tipo de feature")
    imp.groupby(canal.to_numpy()).sum().sort_values().plot.barh(ax=axes[2], title="Por canal")
    fig.tight_layout()

## Honesta vs. K-Fold con fuga
Cuánto se infla el rendimiento si se separan ventanas solapadas del mismo segmento entre entrenamiento y prueba.
Por defecto se muestra el último reporte guardado; con `RUN_DIAGNOSIS = True` se recalcula.

In [ ]:
diag_path = REPORTS / "generalization_diagnosis.md"
if RUN_DIAGNOSIS:
    from src.evaluation.diagnose_generalization import diagnose_generalization
    df, _ = load_dataset(with_raw=False)
    display(diagnose_generalization(df, ("rf", "svm", "knn")).round(3))
elif diag_path.exists():
    display(Markdown(diag_path.read_text(encoding="utf-8")))
else:
    print("Aún no hay diagnóstico: ejecuta python -m scripts.evaluate_models")

## Conclusiones

*(Anota aquí lo que observes: ¿qué clases se confunden y por qué?, ¿algún sujeto es más difícil?, ¿qué canales
importan?, ¿qué haría falta para mejorar: más sujetos, otro montaje, más canales?)*